In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Window
import pyspark.sql.functions as F

CATALOG_SCHEMA  = "teste_koin.default."
BRONZE_TABLE    = "bronze_customers"
SILVER_TABLE    = "silver_customers"
MERGE_KEY       = "customer_id"
DEDUP_ORDER_COL = "ingestion_date"

def assert_columns_exist(df, required_cols: list[str], stage: str) -> None:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{stage}] Colunas ausentes na tabela: {missing}")

df_bronze = spark.table(CATALOG_SCHEMA + BRONZE_TABLE)

REQUIRED_COLUMNS = [
    "customer_id", "name", "email", "phone",
    "city", "state", "status", "created_at", "ingestion_date",
]
assert_columns_exist(df_bronze, REQUIRED_COLUMNS, "bronze")

df_valid_keys = df_bronze.filter(F.col(MERGE_KEY).isNotNull())

# Deduplicação
# Critério: ingestion_date mais recente; em caso de empate, status "blocked" tem prioridade
window_spec = (
    Window
    .partitionBy(MERGE_KEY)
    .orderBy(
        F.col(DEDUP_ORDER_COL).desc(),
        F.when(F.col("status") == "blocked", 1).otherwise(2),
    )
)

df_deduped = (
    df_valid_keys
    .withColumn("_row_number", F.row_number().over(window_spec))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# ATENÇÃO: formatos como dd/MM/yyyy e MM/dd/yyyy são ambíguos seria nescessario validar na origem qual o padrao correto para puxar com exatidao exemplo da data 05/06/2024.
created_at_parsed = F.coalesce(
    F.expr("try_to_date(created_at, 'yyyy-MM-dd')"),
    F.expr("try_to_date(created_at, 'dd/MM/yyyy')"),
    F.expr("try_to_date(created_at, 'yyyy/MM/dd')"),
)

masked_email = F.when(
    F.length(F.trim(F.col("email"))) > 5,
    F.regexp_replace(
        F.trim(F.col("email")),
        r"(^.{3}).*(@.*)",
        "$1****$2",
    ),
).otherwise(F.lit("****@****"))

# Mascaramento de telefone: mantém os 6 primeiros dígitos.
# Remove formatação (parênteses, hífens, espaços) antes de mascarar.
masked_phone = F.concat(
    F.substring(
        F.regexp_replace(F.col("phone"), r"[\(\)\-\s]", ""),
        1, 6,
    ),
    F.lit("****"),
)

silver_timestamp = F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")).cast("timestamp")

df_silver_customers = df_deduped.select(
    F.col("customer_id").cast("string"),
    F.initcap(F.trim(F.col("name"))).alias("customer_name"),
    F.sha2(F.lower(F.trim(F.col("email"))), 256).alias("email_hash"),
    masked_email.alias("masked_email"),
    masked_phone.alias("masked_phone"),
    F.trim(F.col("city")).alias("city"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.col("status"),
    created_at_parsed.alias("created_at_date"),
    F.col("ingestion_date").alias("bronze_at"),
    silver_timestamp.alias("silver_at"),
)

# Prefiri o MERGE ao overwrite completo para:
#   1. Preservar histórico de auditoria da tabela Delta.
#   2. Reduzir custo de I/O em tabelas grandes.
#   3. Evitar janelas sem dados durante a reescrita.

if spark.catalog.tableExists(CATALOG_SCHEMA + SILVER_TABLE):
    delta_table = DeltaTable.forName(spark, CATALOG_SCHEMA + SILVER_TABLE)
    (
        delta_table.alias("target")
        .merge(
            df_silver_customers.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    log("MERGE concluído com sucesso.")

else:
    (
        df_silver_customers.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CATALOG_SCHEMA + SILVER_TABLE)
    )
    log("Tabela Silver criada com sucesso.")